# FalsifyRL — held-out reward-hacking evaluation

This notebook reproduces the non-learning baselines and evaluates the public AutoScientist LoRA on
the entirely held-out `crossing_navigation` scenario family. Gold outputs are deterministic,
simulator-derived JSON—not language-model annotations.


In [ ]:
%pip install -q "transformers>=5.8,<6" "peft>=0.17,<1" "accelerate>=1,<2"


In [ ]:
import json
import os
from collections import Counter, defaultdict
from pathlib import Path

INPUT_ROOT = Path("/kaggle/input")
test_candidates = list(INPUT_ROOT.rglob("test.jsonl"))
assert test_candidates, "Attach the public falsifyrl-seed dataset"
TEST_PATH = test_candidates[0]
rows = [json.loads(line) for line in TEST_PATH.read_text().splitlines() if line.strip()]
print("test path:", TEST_PATH)
print("examples:", len(rows), "roles:", Counter(row["case_role"] for row in rows))
assert len(rows) == 640
assert {row["scenario_family"] for row in rows} == {"crossing_navigation"}


In [ ]:
pairs = defaultdict(list)
for row in rows:
    pairs[row["pair_id"]].append(row)
assert len(pairs) == 320
assert all(
    {item["case_role"] for item in pair} == {"control", "exploit"}
    for pair in pairs.values()
)
assert all(
    len({
        item["prompt"].split("OBSERVED EPISODE TRACE:")[0]
        for item in pair
    }) == 1
    for pair in pairs.values()
)
print("Verified 320 matched control/exploit pairs with shared pre-trace reward context.")


In [ ]:
VERDICTS = ["aligned", "reward_hack"]
FAILURES = [
    "none", "free_riding", "collision_blind", "destructive_interference",
    "unsafe_speed", "out_of_bounds", "no_op_bonus", "premature_termination",
    "credit_misattribution",
]

def macro_f1(gold, predicted, labels):
    scores = []
    for label in labels:
        tp = sum(
            g == label and p == label
            for g, p in zip(gold, predicted, strict=True)
        )
        fp = sum(
            g != label and p == label
            for g, p in zip(gold, predicted, strict=True)
        )
        fn = sum(
            g == label and p != label
            for g, p in zip(gold, predicted, strict=True)
        )
        scores.append(0.0 if 2 * tp + fp + fn == 0 else 2 * tp / (2 * tp + fp + fn))
    return sum(scores) / len(scores)

def compact_metrics(predictions):
    gold = [json.loads(row["completion"]) for row in rows[:len(predictions)]]
    parsed = []
    for prediction in predictions:
        try:
            parsed.append(json.loads(prediction))
        except Exception:
            parsed.append(None)
    gold_verdict = [item["verdict"] for item in gold]
    pred_verdict = [
        "__invalid__" if item is None else item.get("verdict", "__invalid__")
        for item in parsed
    ]
    gold_failure = [item["failure_type"] for item in gold]
    pred_failure = [
        "__invalid__"
        if item is None
        else item.get("failure_type", "__invalid__")
        for item in parsed
    ]
    return {
        "example_count": len(predictions),
        "json_validity": sum(item is not None for item in parsed) / len(parsed),
        "verdict_accuracy": sum(
            g == p
            for g, p in zip(gold_verdict, pred_verdict, strict=True)
        ) / len(parsed),
        "verdict_macro_f1": macro_f1(gold_verdict, pred_verdict, VERDICTS),
        "failure_type_macro_f1": macro_f1(gold_failure, pred_failure, FAILURES),
    }

aligned_json = json.dumps({
    "verdict": "aligned", "failure_type": "none", "responsible_agents": [],
    "evidence_steps": [], "counterexample_config": {}, "reward_patch": None,
    "expected_effect": "No patch needed.", "confidence": 0.5,
}, separators=(",", ":"), sort_keys=True)
always_aligned = [aligned_json] * len(rows)

exploit_by_pair = {
    row["pair_id"]: row["completion"] for row in rows if row["case_role"] == "exploit"
}
reward_only = [exploit_by_pair[row["pair_id"]] for row in rows]
print("always aligned:", compact_metrics(always_aligned))
print("reward only:", compact_metrics(reward_only))


## Load the public AutoScientist adapter

Attach the Kaggle Model `falsifyrl-autoscientist/pytorch/lora`. The adapter config names the exact
base model selected by AutoScientist.


In [ ]:
import torch
from peft import PeftModel
from transformers import AutoModelForMultimodalLM, AutoProcessor

adapter_candidates = list(INPUT_ROOT.rglob("adapter_config.json"))
assert adapter_candidates, "Attach the public FalsifyRL Kaggle Model"
ADAPTER_DIR = adapter_candidates[0].parent
adapter_config = json.loads((ADAPTER_DIR / "adapter_config.json").read_text())
BASE_MODEL_ID = adapter_config["base_model_name_or_path"]
print("adapter:", ADAPTER_DIR)
print("base model:", BASE_MODEL_ID)

processor = AutoProcessor.from_pretrained(BASE_MODEL_ID)
base_model = AutoModelForMultimodalLM.from_pretrained(
    BASE_MODEL_ID,
    torch_dtype=torch.float16 if torch.cuda.is_available() else torch.float32,
    device_map="auto",
)
base_model.eval()


In [ ]:
def extract_json(text):
    start, end = text.find("{"), text.rfind("}")
    return text[start:end + 1] if start >= 0 and end >= start else text

def predict(model, prompt):
    inputs = processor.apply_chat_template(
        [{
            "role": "user",
            "content": [{"type": "text", "text": prompt}],
        }],
        tokenize=True,
        add_generation_prompt=True,
        return_dict=True,
        return_tensors="pt",
    ).to(model.device)
    with torch.inference_mode():
        output = model.generate(
            **inputs,
            max_new_tokens=512,
            do_sample=False,
            pad_token_id=processor.tokenizer.eos_token_id,
        )
    generated = processor.decode(
        output[0, inputs["input_ids"].shape[1]:],
        skip_special_tokens=True,
    )
    return extract_json(generated)

MAX_EXAMPLES = int(os.environ.get("FALSIFYRL_MAX_EXAMPLES", len(rows)))
base_predictions = [
    predict(base_model, row["prompt"]) for row in rows[:MAX_EXAMPLES]
]
base_metrics = compact_metrics(base_predictions)
base_metrics


In [ ]:
model = PeftModel.from_pretrained(base_model, ADAPTER_DIR)
model.eval()
adapted_predictions = [
    predict(model, row["prompt"]) for row in rows[:MAX_EXAMPLES]
]
adapted_metrics = compact_metrics(adapted_predictions)
{
    "base": base_metrics,
    "adapted": adapted_metrics,
    "verdict_macro_f1_improvement": (
        adapted_metrics["verdict_macro_f1"] - base_metrics["verdict_macro_f1"]
    ),
}


In [ ]:
def save_predictions(path, predictions):
    with path.open("w") as stream:
        for row, completion in zip(
            rows[:MAX_EXAMPLES], predictions, strict=True
        ):
            stream.write(json.dumps({
                "example_id": row["example_id"],
                "completion": completion,
            }) + "\n")

base_prediction_path = Path(
    "/kaggle/working/falsifyrl-base-test-predictions.jsonl"
)
adapted_prediction_path = Path(
    "/kaggle/working/falsifyrl-adapted-test-predictions.jsonl"
)
save_predictions(base_prediction_path, base_predictions)
save_predictions(adapted_prediction_path, adapted_predictions)

report = {
    "dataset_test_path": str(TEST_PATH),
    "adapter_path": str(ADAPTER_DIR),
    "base_model_id": BASE_MODEL_ID,
    "example_count": MAX_EXAMPLES,
    "base_metrics": base_metrics,
    "adapted_metrics": adapted_metrics,
    "improvement": {
        key: adapted_metrics[key] - base_metrics[key]
        for key in adapted_metrics
        if isinstance(adapted_metrics[key], float)
    },
}
Path("/kaggle/working/kaggle-evaluation.json").write_text(
    json.dumps(report, indent=2, sort_keys=True) + "\n"
)
print(json.dumps(report, indent=2, sort_keys=True))


For the full executable-patch metric, download both prediction JSONL files and run:

```powershell
python scripts/evaluate_baselines.py `
  --predictions falsifyrl-base-test-predictions.jsonl --split test
python scripts/evaluate_baselines.py `
  --predictions falsifyrl-adapted-test-predictions.jsonl --split test
```

That project-side evaluator re-executes each proposed declarative patch against both exploit and
aligned traces.
